In [20]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import plotly.express as px


In [21]:
dictionary=pd.read_csv('data/data_dictionary.csv')
product_info=pd.read_csv('data/products.csv')
orders= pd.read_csv('data/orders.csv')     
order_items= pd.read_csv('data/order_items.csv')
refund_table=pd.read_csv('data/order_item_refunds.csv')
pageviews=pd.read_csv('data/website_pageviews.csv')
sessions=pd.read_csv('data/website_sessions.csv')

**New Customer Count**

In [22]:
user_orders = orders.groupby("user_id")["order_id"].count().reset_index()
user_orders.columns = ["user_id", "total_orders"]

In [23]:
user_orders["user_type"] = "New User"
user_orders.loc[user_orders["total_orders"] > 1, "user_type"] = "Repeat User"

In [24]:
user_counts =user_orders["user_type"].value_counts()
user_counts

user_type
New User       31105
Repeat User      591
Name: count, dtype: int64

In [25]:
fig = px.pie(values=user_counts.values,names=user_counts.index,title="New vs Repeat Users")
fig.update_traces(textinfo='percent+label')
fig.show()

**INSIGHTS:**

* New users make up about 98% of total users.
* Repeat users are only around 2% of the total user base.
* This shows the business is getting many new customers but very few customers are returning.
* There is a huge gap between new and repeat users.
* This suggests a need to improve customer retention and loyalty programs.

In [30]:
repeat_customers = orders_per_customer[orders_per_customer > 1].count()
repeat_customers

591

In [31]:
total_customers = orders['user_id'].nunique()
total_customers

31696

In [32]:
new_customers_count= total_customers-repeat_customers
new_customers_count

31105

**Overall User Repurchase Rate**

In [29]:
orders_per_customer = orders.groupby('user_id')['order_id'].count()
orders_per_customer

user_id
13        1
20        1
59        1
104       1
147       1
         ..
394231    1
394255    1
394257    1
394268    1
394273    1
Name: order_id, Length: 31696, dtype: int64

In [35]:
repeat_customers = orders_per_customer[orders_per_customer > 1].count()
repeat_customers


591

In [36]:
total_customers = orders['user_id'].nunique()
total_customers


31696

In [37]:
repeat_purchase_rate = (repeat_customers / total_customers) * 100
repeat_purchase_rate

1.8645885916203937

In [38]:
print("Overall Repeat Purchase Rate:", round(repeat_purchase_rate,2), "%")

Overall Repeat Purchase Rate: 1.86 %


**Average Time to Second Purchase / Avg Repurchase Cycle**

In [39]:
orders['created_at'] = pd.to_datetime(orders['created_at'])

In [40]:
orders_sorted = orders.sort_values(['user_id','created_at'])


In [41]:
orders['order_number'] = orders.groupby('user_id').cumcount() + 1

In [42]:
first_orders = orders[orders['order_number'] == 1]
first_orders

,order_id,created_at,website_session_id,user_id,primary_product_id,items_purchased,price_usd,cogs_usd,order_number
0,1,2012-03-19 10:42:46,20,20,1,1,49.99,19.49,1
1,2,2012-03-19 19:27:37,104,104,1,1,49.99,19.49,1
2,3,2012-03-20 06:44:45,147,147,1,1,49.99,19.49,1
3,4,2012-03-20 09:41:45,160,160,1,1,49.99,19.49,1
4,5,2012-03-20 11:28:15,177,177,1,1,49.99,19.49,1
...,...,...,...,...,...,...,...,...,...
32308,32309,2015-03-19 03:58:12,472795,394255,1,1,49.99,19.49,1
32309,32310,2015-03-19 04:10:43,472798,394257,4,1,29.99,9.49,1
32310,32311,2015-03-19 05:27:28,472809,394268,2,2,89.98,31.98,1
32311,32312,2015-03-19 05:35:57,472814,394273,4,1,29.99,9.49,1


In [43]:
second_orders = orders[orders['order_number'] == 2]
second_orders

,order_id,created_at,website_session_id,user_id,primary_product_id,items_purchased,price_usd,cogs_usd,order_number
386,387,2012-06-26 21:13:49,12760,3561,1,1,49.99,19.49,2
499,500,2012-07-19 17:43:50,15845,11071,1,1,49.99,19.49,2
750,751,2012-08-24 13:08:47,21948,16442,1,1,49.99,19.49,2
1063,1064,2012-09-27 10:28:01,29490,18996,1,1,49.99,19.49,2
1174,1175,2012-10-09 20:30:23,32278,28721,1,1,49.99,19.49,2
...,...,...,...,...,...,...,...,...,...
32163,32164,2015-03-17 07:38:24,470902,351605,1,2,95.98,33.98,2
32192,32193,2015-03-17 13:34:09,471198,354670,2,1,59.99,22.49,2
32211,32212,2015-03-17 18:25:47,471494,351985,2,1,59.99,22.49,2
32214,32215,2015-03-17 19:04:42,471524,359445,1,2,79.98,28.98,2


In [44]:
repurchase_data = pd.merge(first_orders, second_orders, on='user_id')
repurchase_data

,order_id_x,created_at_x,website_session_id_x,user_id,primary_product_id_x,items_purchased_x,price_usd_x,cogs_usd_x,order_number_x,order_id_y,created_at_y,website_session_id_y,primary_product_id_y,items_purchased_y,price_usd_y,cogs_usd_y,order_number_y
0,317,2012-06-11 18:56:38,10674,3561,1,1,49.99,19.49,1,387,2012-06-26 21:13:49,12760,1,1,49.99,19.49,2
1,355,2012-06-19 20:11:05,11835,11071,1,1,49.99,19.49,1,500,2012-07-19 17:43:50,15845,1,1,49.99,19.49,2
2,592,2012-08-02 10:19:15,17815,16442,1,1,49.99,19.49,1,751,2012-08-24 13:08:47,21948,1,1,49.99,19.49,2
3,703,2012-08-18 22:53:48,20652,18996,1,1,49.99,19.49,1,1064,2012-09-27 10:28:01,29490,1,1,49.99,19.49,2
4,821,2012-09-04 10:15:24,24121,22180,1,1,49.99,19.49,1,1528,2012-11-06 15:20:39,39909,1,1,49.99,19.49,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
586,30558,2015-02-21 08:08:02,451792,377703,4,1,29.99,9.49,1,31082,2015-03-01 18:30:39,458130,1,1,49.99,19.49,2
587,30659,2015-02-23 14:11:56,453033,351605,1,2,79.98,28.98,1,32164,2015-03-17 07:38:24,470902,1,2,95.98,33.98,2
588,31349,2015-03-05 06:01:38,461351,322595,1,1,49.99,19.49,1,31981,2015-03-13 21:06:37,468894,1,1,49.99,19.49,2
589,31424,2015-03-06 03:54:04,462329,366677,1,1,49.99,19.49,1,32005,2015-03-14 11:35:55,469156,1,1,49.99,19.49,2


In [45]:
repurchase_data['created_at_x'] = pd.to_datetime(repurchase_data['created_at_x'])
repurchase_data['created_at_y'] = pd.to_datetime(repurchase_data['created_at_y'])
repurchase_data['days_between'] = (
    repurchase_data['created_at_y'] - repurchase_data['created_at_x']
).dt.days

In [46]:
average_days = repurchase_data['days_between'].mean()
print("Average Time to Second Purchase:", round(average_days,2), "days")

Average Time to Second Purchase: 34.68 days


In [47]:
fig = px.histogram(
    repurchase_data,
    x="days_between",
    nbins=40,
    title="Distribution of Time to Second Purchase"
)

fig.show()

**Insights:**
- The visualization highlights differences across categories or time periods.
- Certain segments contribute more significantly to overall performance.
- Noticeable trends indicate periods of higher or lower activity.
- These patterns help support data‑driven decisions and improve business understanding.


**Repeat customer AOV**

In [48]:
order_counts = orders.groupby('user_id')['order_id'].count()
repeat_customers = order_counts[order_counts > 1].index


In [49]:
repeat_orders = orders[orders['user_id'].isin(repeat_customers)]
repeat_aov = repeat_orders['price_usd'].sum() / repeat_orders['order_id'].count()

print("Repeat Customer AOV:", round(repeat_aov,2))

Repeat Customer AOV: 61.55


Order Value Distribution

In [50]:
 orders_with_type = orders.merge(
    user_orders[['user_id', 'user_type']],
    on='user_id',
    how='left'
)
orders_with_type

,order_id,created_at,website_session_id,user_id,primary_product_id,items_purchased,price_usd,cogs_usd,order_number,user_type
0,1,2012-03-19 10:42:46,20,20,1,1,49.99,19.49,1,New User
1,2,2012-03-19 19:27:37,104,104,1,1,49.99,19.49,1,New User
2,3,2012-03-20 06:44:45,147,147,1,1,49.99,19.49,1,New User
3,4,2012-03-20 09:41:45,160,160,1,1,49.99,19.49,1,New User
4,5,2012-03-20 11:28:15,177,177,1,1,49.99,19.49,1,New User
...,...,...,...,...,...,...,...,...,...,...
32308,32309,2015-03-19 03:58:12,472795,394255,1,1,49.99,19.49,1,New User
32309,32310,2015-03-19 04:10:43,472798,394257,4,1,29.99,9.49,1,New User
32310,32311,2015-03-19 05:27:28,472809,394268,2,2,89.98,31.98,1,New User
32311,32312,2015-03-19 05:35:57,472814,394273,4,1,29.99,9.49,1,New User


In [51]:
import plotly.express as px

fig = px.violin(
    orders_with_type,
    x="user_type",
    y="price_usd",
    box=False,         
    points="all",     
    title="Order Value Distribution: New vs Repeat Customers"
)

fig.update_layout(
    xaxis_title="Customer Type",
    yaxis_title="Order Value (USD)",
    title_x=0.5
)

fig.show()

**Insights:**
- The visualization highlights differences across categories or time periods.
- Certain segments contribute more significantly to overall performance.
- Noticeable trends indicate periods of higher or lower activity.
- These patterns help support data‑driven decisions and improve business understanding.


**Repeat purchase rate per product**

In [52]:
df = order_items.merge(
    orders[["order_id","user_id"]],
    on="order_id",
    how="left"
).merge(
    product_info[["product_id","product_name"]],
    on="product_id",
    how="left"
)

In [53]:
user_product = df.groupby(["product_name","user_id"]).size().reset_index(name="times_bought")

In [54]:
total_customers = user_product.groupby("product_name")["user_id"].nunique().reset_index(name="total_customers")

In [55]:
repeat_df = user_product[user_product["times_bought"] > 1]

In [56]:
repeat_customers = repeat_df.groupby("product_name")["user_id"].nunique()

In [57]:
repeat_customers = repeat_customers.reset_index()
repeat_customers.columns = ["product_name", "repeat_customers"]

In [58]:
result = total_customers.merge(repeat_customers, on="product_name", how="left").fillna(0)
result["repeat_purchase_rate_%"] = (result["repeat_customers"] / result["total_customers"]) * 100
result

,product_name,total_customers,repeat_customers,repeat_purchase_rate_%
0,The Birthday Sugar Panda,4957,27,0.544684
1,The Forever Love Bear,5776,20,0.346260
2,The Hudson River Mini bear,4982,36,0.722601
3,The Original Mr. Fuzzy,23887,326,1.364759


In [59]:
fig = px.bar(
    result,
    x="product_name",
    y="repeat_purchase_rate_%",
    title="Actual Repeat Purchase Rate per Product"
)

fig.update_layout(
    xaxis_title="Product Name",
    yaxis_title="Repeat Purchase Rate (%)"
)

fig.show()

**Insights:**
- The visualization highlights differences across categories or time periods.
- Certain segments contribute more significantly to overall performance.
- Noticeable trends indicate periods of higher or lower activity.
- These patterns help support data‑driven decisions and improve business understanding.


In [60]:
def repurchase_rate(data):
    fig = px.bar(
    data,
    x="product_name",
    y="repeat_purchase_rate_%",
    title="Actual Repeat Purchase Rate per Product"
    )

    fig.update_layout(
        xaxis_title="Product Name",
        yaxis_title="Repeat Purchase Rate (%)"
    )
    return fig

In [61]:
repurchase_rate(result)

In [62]:
def order_val_dist(data):
    fig = px.violin(
    data,
    x="user_type",
    y="price_usd",
    box=False,         
    points="all",     
    title="Order Value Distribution: New vs Repeat Customers"
    )

    fig.update_layout(
        xaxis_title="Customer Type",
        yaxis_title="Order Value (USD)",
        title_x=0.5
    )
    return fig

In [63]:
order_val_dist(orders_with_type)

In [64]:
def dist_time(data):
    fig = px.histogram(
    data,
    x="days_between",
    nbins=40,
    title="Distribution of Time to Second Purchase"
    )

    return fig

In [65]:
dist_time(repurchase_data)

In [68]:
def new_vs_repeat(data):
    fig = px.pie(values=data,names=user_counts.index,title="New vs Repeat Users")
    fig.update_traces(textinfo='percent+label')
    return fig

In [69]:
new_vs_repeat(user_counts.values)